## Environment setup

Installs the metric libraries used to inspect source code and restarts the Python runtime.


In [0]:
# This setup cell prepares the runtime with the static-analysis libraries used below.

%pip install radon==6.0.1 lizard==1.17.10
dbutils.library.restartPython()

## Imports, configuration, and parsing patterns

Defines dependencies, pipeline roots, DQ-specific regex patterns, helper metadata functions, and chart colors.


In [0]:
# This configuration cell defines the project scope, DQ detection patterns, and reusable metadata helpers.

# Import dependencies and define the global notebook configuration.
import os
import re
import ast
import textwrap
import difflib
from collections import defaultdict, Counter
from radon.complexity import cc_visit
from radon.metrics import mi_visit
from radon.raw import analyze as raw_analyze
import matplotlib.pyplot as plt
import numpy as np

import lizard
import pandas as pd

# Root folders for both pipeline implementations in the Databricks workspace.
DECLARATIVE_ROOT = "/Workspace/Users/klementf.wwi23@student.dhbw-heidenheim.de/thesis-databricks-pipeline-poc/declarative_pipeline/transformations"
IMPERATIVE_ROOT  = "/Workspace/Users/klementf.wwi23@student.dhbw-heidenheim.de/thesis-databricks-pipeline-poc/imperative_pipeline"

PIPELINES = {
    "declarative": DECLARATIVE_ROOT,
    "imperative": IMPERATIVE_ROOT,
}

# Subfolder that contains the data quality implementation.
DQ_SUBPATH = "data_quality"

# Regex patterns used to identify data-quality-specific lines of code.
DQ_PATTERNS = {
    "declarative": [
        r'dlt\.read\("dq_rule_catalog"\)',
        r'catalog_rule\(',
        r'\bexpect\b',
        r'\bEXPECT\b',
        r'\bMONITOR\b',
        r'aggregate_metrics',
        r'enrich_with_rule',
    ],
    "imperative": [
        r'issue_frame\(',
        r'metrics_frame\(',
        r'count_if\(',
        r'pct_null\(',
        r'duplicate_row_count\(',
        r'\.isNull\(\)',
        r'\.isNotNull\(\)',
        r'F\.when\(',
    ],
}

# Function-name hints used to identify reusable data quality helpers.
DQ_FUNC_NAME_HINTS = [
    "dq", "quality", "issue_frame", "metrics_frame", "rule", "monitor"
]


# Helper functions for deriving pipeline metadata from file paths.

def detect_layer_from_filename(path: str) -> str:
    base = os.path.basename(path)
    if base.startswith("01_"):
        return "bronze"
    elif base.startswith("02_"):
        return "silver"
    elif base.startswith("03_"):
        return "gold"
    else:
        return "catalog"


def is_data_quality(path: str) -> bool:
    return "data_quality" in path.replace("\\", "/")


def detect_job_name(path: str) -> str:
    base = os.path.basename(path)
    name, _ = os.path.splitext(base)
    return name
# colors for figures
declarative_color = "#0F2DB3"
imperative_color = "#0072EF"


## Data quality file discovery and parsing helpers

Adds reusable helpers for listing files, reading Databricks workspace files, counting DQ lines, and detecting duplicated call blocks.


In [0]:
# These helpers abstract file access and text parsing so the KPI cells can focus on metric logic.

# Helper functions for file discovery, reading, parsing, and duplicate detection.
# These helpers abstract file access and text parsing so the KPI cells can focus on metric logic.

import os
import re
import difflib


# Helper functions for file discovery, reading, parsing, and duplicate detection.
def _candidate_dq_paths(root: str, subpath: str) -> list[str]:
    """
    Build a list of plausible DQ directories.

    This is necessary because the declarative pipeline may store DQ files under
    .../transformations/data_quality instead of directly under .../data_quality.
    """
    root = root.rstrip("/")

    candidates = [
        os.path.join(root, subpath),
        os.path.join(root, "data_quality"),
        os.path.join(root, "transformations", "data_quality"),
        os.path.join(root, "transformations", subpath),
    ]

    seen = set()
    ordered = []
    for c in candidates:
        c_norm = c.replace("\\", "/")
        if c_norm not in seen:
            seen.add(c_norm)
            ordered.append(c_norm)
    return ordered


def _dbutils_path_variants(path: str) -> list[str]:
    """
    Build path variants for Databricks workspace access.
    """
    path = path.replace("\\", "/").rstrip("/")

    variants = [path]

    if path.startswith("/Workspace"):
        variants.append("dbfs:" + path)
    elif path.startswith("dbfs:/Workspace"):
        variants.append(path.replace("dbfs:", "", 1))

    seen = set()
    ordered = []
    for v in variants:
        if v not in seen:
            seen.add(v)
            ordered.append(v)
    return ordered


def list_py_files(root: str, subpath: str):
    all_files = []

    # Try multiple plausible base directories
    for base in _candidate_dq_paths(root, subpath):
        found_here = []

        # 1) Try Databricks filesystem variants
        for db_path in _dbutils_path_variants(base):
            try:
                def _walk_dbutils(path):
                    for f in dbutils.fs.ls(path):
                        if f.isDir():
                            _walk_dbutils(f.path)
                        elif f.path.endswith(".py"):
                            found_here.append(f.path)

                _walk_dbutils(db_path)
                if found_here:
                    all_files.extend(found_here)
                    break
            except Exception:
                pass

        # 2) Fallback: local filesystem
        if not found_here:
            local_candidates = [base]

            if base.startswith("dbfs:/"):
                local_candidates.append(base.replace("dbfs:/", "/dbfs/", 1))
            elif base.startswith("/Workspace"):
                local_candidates.append(base)
                local_candidates.append("/Workspace" + base[len("/Workspace"):])

            for local_base in local_candidates:
                if os.path.isdir(local_base):
                    for dirpath, _, filenames in os.walk(local_base):
                        for fn in filenames:
                            if fn.endswith(".py"):
                                found_here.append(os.path.join(dirpath, fn))
                    if found_here:
                        all_files.extend(found_here)
                        break

        # stop early once a candidate path works
        if found_here:
            break

    # de-duplicate + sort
    normalized = sorted(set(f.replace("\\", "/") for f in all_files))
    return normalized


def read_file(path: str, max_bytes: int = 1_000_000) -> str:
    path = path.replace("\\", "/")

    # 1) Try Databricks filesystem variants
    for p in _dbutils_path_variants(path):
        try:
            return dbutils.fs.head(p, max_bytes)
        except Exception:
            pass

    # 2) Fallback: local filesystem
    local_candidates = [path]

    if path.startswith("dbfs:/"):
        local_candidates.append(path.replace("dbfs:/", "/dbfs/", 1))
    elif path.startswith("/Workspace"):
        local_candidates.append(path)

    for local_path in local_candidates:
        try:
            with open(local_path, "r", encoding="utf-8") as f:
                return f.read(max_bytes)
        except Exception:
            pass

    print(f"[WARN] could not read file: {path}")
    return ""


def non_empty_code_lines(text: str):
    lines = []
    for line in text.splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        lines.append(line)
    return lines


def count_dq_loc(lines, patterns):
    count = 0
    for line in lines:
        if any(re.search(p, line) for p in patterns):
            count += 1
    return count


def detect_reusable_functions(text: str):
    func_names = []
    for m in re.finditer(r"^\s*def\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(", text, flags=re.M):
        name = m.group(1)
        if any(hint in name.lower() for hint in DQ_FUNC_NAME_HINTS):
            func_names.append(name)
    return func_names


def normalize_code_for_clone(block: str) -> str:
    block = re.sub(r"\".*?\"|'.*?'", "STR", block)
    block = re.sub(r"\b\d+\b", "NUM", block)
    block = re.sub(r"\s+", " ", block).strip()
    return block


def extract_call_blocks(text: str, fn_names):
    blocks = []
    for fn in fn_names:
        for m in re.finditer(rf"{re.escape(fn)}\s*\((.*?)\)", text, flags=re.S):
            blocks.append(m.group(0))
    return blocks


def detect_duplicate_blocks(blocks, threshold: float = 0.8) -> int:
    dups = 0
    used = set()
    for i in range(len(blocks)):
        for j in range(i + 1, len(blocks)):
            if (i, j) in used:
                continue
            a = normalize_code_for_clone(blocks[i])
            b = normalize_code_for_clone(blocks[j])
            if difflib.SequenceMatcher(None, a, b).ratio() >= threshold:
                dups += 1
                used.add((i, j))
    return dups


# Data quality files grouped by implementation paradigm.
DQ_FILES = {
    paradigm: list_py_files(root, DQ_SUBPATH)
    for paradigm, root in PIPELINES.items()
}

print("DQ_FILES:", DQ_FILES)
DQ_FILES

## Discover pipeline source files

Scans both pipeline implementations and annotates every Python file with layer, job, and data quality metadata.


In [0]:
# This cell creates the source-file inventory used as the basis for all data quality KPIs.

# Collect all Python files from both pipelines and enrich them with metadata.

records = []

for pipeline_type, root in PIPELINES.items():
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if not fname.endswith(".py"):
                continue

            full_path = os.path.join(dirpath, fname)

            records.append({
                "pipeline": pipeline_type,
                "file_path": full_path,
                "layer": detect_layer_from_filename(full_path),
                "job": detect_job_name(full_path),
                "is_data_quality": is_data_quality(full_path),
            })

df_files = pd.DataFrame(records)

print("All discovered .py files:")
display(df_files)

print("Business transformation jobs only, excluding data_quality:")
df_main_jobs = df_files[~df_files["is_data_quality"]].reset_index(drop=True)
display(df_main_jobs)

## Radon file analysis helper

Encapsulates SLOC, complexity, and maintainability calculations for one Python file.


In [0]:
# This function normalizes Radon's raw outputs into a compact dictionary of file-level metrics.

# Helper function for analyzing one Python file with Radon.

def analyze_python_file(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        code = f.read()

    # Raw metrics (inkl. SLOC)
    raw = raw_analyze(code)
    sloc = raw.sloc

    # Cyclomatic Complexity
    cc_results = cc_visit(code)
    if cc_results:
        cc_scores = [item.complexity for item in cc_results]
        avg_cc = sum(cc_scores) / len(cc_scores)
        max_cc = max(cc_scores)
        cc_count = len(cc_scores)
    else:
        avg_cc = 0.0
        max_cc = 0.0
        cc_count = 0

    # Maintainability Index
    mi_results = mi_visit(code, multi=True)

    # mi_results kann ein Float oder eine Sequenz sein
    mi_scores = []
    if isinstance(mi_results, (int, float)):
        mi_scores = [float(mi_results)]
    elif mi_results:
        # Radon gibt hier eine Sequenz von Objekten mit Attribut .mi zurück
        # oder direkt eine Sequenz von Floats, je nach Version
        first = mi_results[0]
        if hasattr(first, "mi"):
            mi_scores = [r.mi for r in mi_results]
        else:
            mi_scores = [float(r) for r in mi_results]

    if mi_scores:
        avg_mi = sum(mi_scores) / len(mi_scores)
        min_mi = min(mi_scores)
        mi_count = len(mi_scores)
    else:
        avg_mi = 100.0
        min_mi = 100.0
        mi_count = 0

    return {
        "sloc": sloc,
        "avg_cc": avg_cc,
        "max_cc": max_cc,
        "cc_entities": cc_count,
        "avg_mi": avg_mi,
        "min_mi": min_mi,
        "mi_entities": mi_count,
    }

## KPI 1: implementation effort for DQ checks

Calculates DQ-specific SLOC by paradigm and layer, then aggregates the same metric at pipeline level.


In [0]:
# This KPI measures how much of each paradigm's code base is dedicated to data quality logic.

# KPI 1 – Implementation effort for data quality checks.

metrics_records = []

for _, row in df_files.iterrows():
    path = row["file_path"]
    try:
        metrics = analyze_python_file(path)  # liefert u.a. "sloc"
    except Exception as e:
        print(f"Error while analyzing {path}: {e}")
        continue

    record = {
        "pipeline": row["pipeline"],
        "layer": row["layer"],
        "job": row["job"],
        "file_path": row["file_path"],
        "is_data_quality": row["is_data_quality"],
    }
    record.update(metrics)  # hängt "sloc", "avg_cc", ... an
    metrics_records.append(record)

df_metrics_file = pd.DataFrame(metrics_records)

print("File-level Radon metrics:")
display(df_metrics_file)

# 2) KPI 1 – Implementation Effort for Data Quality Checks
#    Aggregation: by paradigm (pipeline) and layer (bronze/silver/gold).

kpi1_rows = []

# Group by pipeline and layer and aggregate across all files.
grouped = df_metrics_file.groupby(["pipeline", "layer"], dropna=False)

for (pipeline, layer), group in grouped:
    # Total SLOC of all files in this paradigm and layer.
    total_sloc = group["sloc"].sum()

    # SLOC of data quality files only in this paradigm and layer.
    dq_sloc = group.loc[group["is_data_quality"], "sloc"].sum()

    dq_percentage = (dq_sloc / total_sloc * 100.0) if total_sloc > 0 else 0.0

    # Optional counts for all files and data quality files.
    num_files = group["file_path"].nunique()
    num_dq_files = group.loc[group["is_data_quality"], "file_path"].nunique()

    kpi1_rows.append(
        {
            "pipeline": pipeline,
            "layer": layer,
            "total_sloc": total_sloc,
            "dq_sloc": dq_sloc,
            "dq_sloc_percentage": dq_percentage,
            "num_files": num_files,
            "num_dq_files": num_dq_files,
        }
    )

kpi1_df = pd.DataFrame(kpi1_rows).sort_values(["pipeline", "layer"])
print("KPI 1 – SLOC overview by paradigm and layer, based on Radon SLOC:")
display(kpi1_df)

# 3) Optional paradigm-only view aggregated across all layers.

kpi1_pipeline_rows = []

pipeline_grouped = df_metrics_file.groupby("pipeline", dropna=False)

for pipeline, group in pipeline_grouped:
    total_sloc = group["sloc"].sum()
    dq_sloc = group.loc[group["is_data_quality"], "sloc"].sum()
    dq_percentage = (dq_sloc / total_sloc * 100.0) if total_sloc > 0 else 0.0

    num_files = group["file_path"].nunique()
    num_dq_files = group.loc[group["is_data_quality"], "file_path"].nunique()

    kpi1_pipeline_rows.append(
        {
            "pipeline": pipeline,
            "total_sloc": total_sloc,
            "dq_sloc": dq_sloc,
            "dq_sloc_percentage": dq_percentage,
            "num_files": num_files,
            "num_dq_files": num_dq_files,
        }
    )

kpi1_pipeline_df = pd.DataFrame(kpi1_pipeline_rows).sort_values("pipeline")
print("KPI 1 – SLOC overview by paradigm across all layers:")
display(kpi1_pipeline_df)

## KPI 2: rule reusability and duplication

Extracts rule IDs and duplicated DQ call blocks to compare reuse between declarative and imperative implementations.


In [0]:
# This KPI compares rule reuse and duplicated implementation patterns across both paradigms.

# KPI 2 – Reusability of data quality rules.
import re
from collections import Counter

# ---------- Imperative paradigm: extract rule IDs from data quality files ----------

def parse_imp_rule_ids(text: str) -> list[str]:
    """
    Extract all rule IDs from imperative data quality files.

    Sources:
    - issue_frame/metrics_frame("table", "IP_SILVER_010", ...)
    - F.lit("IP_SILVER_013").alias("rule_id")
    - Optional DP_SILVER_xxx values if they are written as rule IDs in quarantine logic.
    """
    rule_ids: list[str] = []

    # issue_frame/metrics_frame("table", "RULE_ID", ...)
    pattern_fn = (
        r'(?:issue_frame|metrics_frame)\s*'
        r'\(\s*["\'].*?["\']\s*,\s*["\']([^"\']+)["\']'
    )
    rule_ids += re.findall(pattern_fn, text)

    # F.lit("RULE_ID").alias("rule_id")
    pattern_lit = (
        r'F\.lit\(\s*["\']([^"\']+)["\']\s*\)\.alias\(\s*["\']rule_id["\']\s*\)'
    )
    rule_ids += re.findall(pattern_lit, text)

    return rule_ids


imp_rule_ids: list[str] = []
imp_call_blocks: list[str] = []

for path in DQ_FILES["imperative"]:
    text = read_file(path)
    if not text:
        continue

    imp_rule_ids.extend(parse_imp_rule_ids(text))

    # Capture central data quality calls for duplicate-block analysis.
    imp_call_blocks.extend(
        extract_call_blocks(text, ["issue_frame", "metrics_frame", "build_store_sales_quarantine"])
    )

imp_rule_counter = Counter(imp_rule_ids)
imp_unique_rules = len(imp_rule_counter)
imp_total_instances = sum(imp_rule_counter.values())
imp_duplicate_instances = sum((c - 1) for c in imp_rule_counter.values() if c > 1)

imp_reuse_ratio = (
    imp_total_instances / imp_unique_rules if imp_unique_rules > 0 else 0.0
)
imp_duplicate_instances_ratio = (
    imp_duplicate_instances / imp_unique_rules if imp_unique_rules > 0 else 0.0
)

imp_duplicate_blocks = detect_duplicate_blocks(imp_call_blocks, threshold=0.8)
imp_duplicate_block_ratio = (
    imp_duplicate_blocks / imp_unique_rules if imp_unique_rules > 0 else 0.0
)


# ---------- Declarative paradigm: extract catalog rule IDs from data quality files ----------

def parse_decl_rule_ids(text: str) -> list[str]:
    """
    Extract catalog rule IDs such as BRZ_*, DP_SILVER_*, and DP_GOLD_* from declarative data quality files.

    Sources:
    - catalog_rule("DP_SILVER_010")
    - F.col("rule_id") == "DP_SILVER_010"
    - F.lit("DP_SILVER_010").alias("rule_id")
    - Optional DQ-specific DLT config files with @dlt.expect*("<ID>", ...).
      Only catalog-like IDs are counted, not free-text expectation names.
    """
    rule_ids: list[str] = []

    # 1) catalog_rule("RULE_ID")
    rule_ids += re.findall(
        r'catalog_rule\(\s*["\']([^"\']+)["\']',
        text,
    )

    # 2) F.col("rule_id") == "RULE_ID"
    rule_ids += re.findall(
        r'F\.col\("rule_id"\)\s*==\s*["\']([^"\']+)["\']',
        text,
    )

    # 3) F.lit("RULE_ID").alias("rule_id")
    rule_ids += re.findall(
        r'F\.lit\(\s*["\']([^"\']+)["\']\s*\)\.alias\(\s*["\']rule_id["\']\s*\)',
        text,
    )

    # 4) Optional: DLT expect decorators inside data quality files, not transformation code.
    dlt_ids = re.findall(
        r'@dlt\.expect[_a-z]*\(\s*["\']([^"\']+)["\']',
        text,
    )
    for rid in dlt_ids:
        if re.match(r'^[A-Z0-9_]+$', rid):
            rule_ids.append(rid)

    return rule_ids


decl_rule_ids: list[str] = []
decl_call_blocks: list[str] = []

for path in DQ_FILES["declarative"]:
    text = read_file(path)
    if not text:
        continue

    decl_rule_ids.extend(parse_decl_rule_ids(text))

    # Capture catalog_rule and DQ-related DLT calls for duplicate-block analysis.
    decl_call_blocks.extend(
        extract_call_blocks(text, ["catalog_rule", "@dlt.expect", "@dlt.expect_or_drop"])
    )

decl_rule_counter = Counter(decl_rule_ids)
decl_unique_rules = len(decl_rule_counter)
decl_total_refs = sum(decl_rule_counter.values())
decl_duplicate_instances = sum((c - 1) for c in decl_rule_counter.values() if c > 1)

decl_reuse_ratio = (
    decl_total_refs / decl_unique_rules if decl_unique_rules > 0 else 0.0
)
decl_duplicate_instances_ratio = (
    decl_duplicate_instances / decl_unique_rules if decl_unique_rules > 0 else 0.0
)

decl_duplicate_blocks = detect_duplicate_blocks(decl_call_blocks, threshold=0.8)
decl_duplicate_block_ratio = (
    decl_duplicate_blocks / decl_unique_rules if decl_unique_rules > 0 else 0.0
)


# ---------- KPI 2 result DataFrame ----------

kpi2_rows = [
    {
        "pipeline": "declarative",
        "unique_rules": decl_unique_rules,
        "total_rule_applications": decl_total_refs,
        "reuse_ratio": decl_reuse_ratio,
        "duplicate_instances": decl_duplicate_instances,
        "duplicate_instance_ratio": decl_duplicate_instances_ratio,
        "duplicate_blocks": decl_duplicate_blocks,
        "duplicate_block_ratio": decl_duplicate_block_ratio,
    },
    {
        "pipeline": "imperative",
        "unique_rules": imp_unique_rules,
        "total_rule_applications": imp_total_instances,
        "reuse_ratio": imp_reuse_ratio,
        "duplicate_instances": imp_duplicate_instances,
        "duplicate_instance_ratio": imp_duplicate_instances_ratio,
        "duplicate_blocks": imp_duplicate_blocks,
        "duplicate_block_ratio": imp_duplicate_block_ratio,
    },
]

kpi2_df = pd.DataFrame(kpi2_rows)
display(kpi2_df)

## KPI 3: standardization across Silver and Gold

Checks whether Silver and Gold tables satisfy minimum rule-group standards in each paradigm.


In [0]:
# This KPI checks whether each paradigm applies a consistent minimum set of rules in Silver and Gold.

# KPI 3 – Standardization across layers, limited to Silver and Gold.

from collections import Counter

MIN_STANDARDS = {
    "silver": {
        "KEY_VALIDITY": 2,
        "BUSINESS_VALUE": 1,
    },
    "gold": {
        "BUSINESS_KEY": 1,
        "MEASURE_VALIDITY": 1,
    },
}


def layer_from_table_name(table_name: str) -> str:
    name = table_name.lower()
    if name.startswith("silver_"):
        return "silver"
    if name.startswith("gold_"):
        return "gold"
    # Explicitly ignore Bronze and other non-target layers.
    return "other"


def check_standard_for_table(layer: str, rule_groups: list[str]) -> bool:
    required = MIN_STANDARDS.get(layer, {})
    counts = Counter(rule_groups)
    for group, min_count in required.items():
        if counts.get(group, 0) < min_count:
            return False
    return True


# ---------- Declarative rules: read from dp_rule_catalog_pub ----------

try:
    decl_catalog_df = spark.table("workspace.declarative.dp_rule_catalog_pub")
    # Keep only Silver and Gold rules for this KPI.
    decl_catalog_filt = decl_catalog_df.where("layer IN ('silver','gold')")
    decl_rules_pd = decl_catalog_filt.select("layer", "object_name", "rule_group").toPandas()
except Exception as e:
    print("[WARN] Could not read dp_rule_catalog_pub; declarative KPI 3 remains empty:", e)
    decl_rules_pd = pd.DataFrame(columns=["layer", "object_name", "rule_group"])

decl_kpi_rows = []
if not decl_rules_pd.empty:
    # Keep only Silver and Gold rows as a safety filter.
    decl_rules_pd = decl_rules_pd[decl_rules_pd["layer"].isin(["silver", "gold"])]
    for (layer, object_name), grp in decl_rules_pd.groupby(["layer", "object_name"]):
        rule_groups = grp["rule_group"].tolist()
        meets = check_standard_for_table(layer, rule_groups)
        num_rules = len(rule_groups)
        decl_kpi_rows.append({
            "paradigm": "declarative",
            "layer": layer,
            "table_name": object_name,
            "meets_standard": meets,
            "num_checks": num_rules,
        })


# ---------- Imperative rules: issue_frame coverage for Silver and Gold ----------

def parse_imperative_issue_frame_calls(text: str):
    """
    Parse issue_frame("table_name", "RULE_ID", "RULE_GROUP", ...) calls from the Silver and Gold quality files.
    """
    rows = []
    pattern = r'issue_frame\(\s*["\']([^"\']+)["\']\s*,\s*["\']([^"\']+)["\']\s*,\s*["\']([^"\']+)["\']'
    for m in re.finditer(pattern, text):
        table_name = m.group(1)
        rule_id = m.group(2)
        rule_group = m.group(3)
        rows.append({
            "table_name": table_name,
            "rule_id": rule_id,
            "rule_group": rule_group,
        })
    return rows


imp_issue_rows = []

# Consider only Silver and Gold quality files in the imperative paradigm.
for path in DQ_FILES["imperative"]:
    if not any(x in path for x in ["silver_quality", "gold_quality"]):
        continue
    text = read_file(path)
    if not text:
        continue
    imp_issue_rows.extend(parse_imperative_issue_frame_calls(text))

if imp_issue_rows:
    imp_rules_pd = pd.DataFrame(imp_issue_rows)
else:
    imp_rules_pd = pd.DataFrame(columns=["table_name", "rule_id", "rule_group"])

# Derive the layer from table names and filter out Bronze or other layers.
imp_kpi_rows = []
if not imp_rules_pd.empty:
    imp_rules_pd["layer"] = imp_rules_pd["table_name"].apply(layer_from_table_name)
    imp_rules_pd = imp_rules_pd[imp_rules_pd["layer"].isin(["silver", "gold"])]

    for (layer, table_name), grp in imp_rules_pd.groupby(["layer", "table_name"]):
        rule_groups = grp["rule_group"].tolist()
        meets = check_standard_for_table(layer, rule_groups)
        num_checks = len(rule_groups)
        imp_kpi_rows.append({
            "paradigm": "imperative",
            "layer": layer,
            "table_name": table_name,
            "meets_standard": meets,
            "num_checks": num_checks,
        })


# ---------- Aggregate table-level results into layer-level KPI values ----------

all_rows = decl_kpi_rows + imp_kpi_rows

if all_rows:
    per_table_df = pd.DataFrame(all_rows)
else:
    per_table_df = pd.DataFrame(
        columns=[
            "paradigm",
            "layer",
            "table_name",
            "meets_standard",
            "num_checks",
        ]
    )

kpi3_rows = []
if not per_table_df.empty:
    for (paradigm, layer), grp in per_table_df.groupby(["paradigm", "layer"]):
        total_tables = grp["table_name"].nunique()
        tables_meeting = grp[grp["meets_standard"]]["table_name"].nunique()
        avg_checks = grp["num_checks"].mean() if total_tables > 0 else 0.0
        kpi3_rows.append({
            "paradigm": paradigm,
            "layer": layer,
            "total_tables": total_tables,
            "tables_meeting_standard": tables_meeting,
            "coverage_percentage": (tables_meeting / total_tables * 100.0) if total_tables > 0 else 0.0,
            "avg_checks_per_table": avg_checks,
        })

if kpi3_rows:
    kpi3_pdf = pd.DataFrame(kpi3_rows)
else:
    kpi3_pdf = pd.DataFrame(
        columns=[
            "paradigm",
            "layer",
            "total_tables",
            "tables_meeting_standard",
            "coverage_percentage",
            "avg_checks_per_table",
        ]
    )

# Use Spark only when data exists; otherwise display the empty Pandas fallback.
if not kpi3_pdf.empty:
    kpi3_df = spark.createDataFrame(kpi3_pdf)
    display(kpi3_df)
else:
    print("KPI 3: no Silver or Gold rules found; check the catalog or issue_frame parsing.")
    display(kpi3_pdf)

## KPI 4: effort to evolve DQ rules

Estimates the change impact for selected rule-evolution scenarios across both paradigms.


In [0]:
# This KPI estimates how many files, touch points, and lines change for representative rule updates.

# KPI 4 – Effort to evolve data quality rules, adapted to the project files.

def avg_loc_per_check_imperative():
    """
    Estimate the average lines of code per data quality check in imperative code.
    The estimate is based on issue_frame and bronze_metrics usage.
    """
    loc_counts = []
    for path in DQ_FILES["imperative"]:
        text = read_file(path)
        if not text:
            continue
        lines = non_empty_code_lines(text)
        dq_loc = count_dq_loc(lines, DQ_PATTERNS["imperative"])
        num_calls = (
            len(re.findall(r"issue_frame\s*\(", text)) +
            len(re.findall(r"metrics_frame\s*\(", text)) +
            len(re.findall(r"bronze_metrics\s*\(", text))
        )
        if num_calls > 0:
            loc_counts.append(dq_loc / num_calls)
    return sum(loc_counts) / len(loc_counts) if loc_counts else 10.0


def change_impact_add_rule_all_customers():
    """
    Scenario: add a new 'email_format_valid' rule to all customer tables
    (bronze_customer, silver_customer, gold_dim_customer).

    Declarative approach:
      - Add the rule once to the central dq_rule_catalog file.

    Imperative approach:
      - Add customer-specific issue_frame logic in every quality file that contains customer tables.
    """
    # Declarative approach: central rule catalog.
    decl_catalog_files = [f for f in DQ_FILES["declarative"] if "dq_rule_catalog" in f or "rule_catalog" in f]
    if not decl_catalog_files and DQ_FILES["declarative"]:
        decl_catalog_files = [DQ_FILES["declarative"][0]]

    decl_files_to_modify = len(decl_catalog_files)
    decl_touch_points = 1  # eine neue Rule in RULES
    decl_loc_change = 3    # grob 3 LOC (neuer Dict-Eintrag)

    # Imperative approach: all files that contain customer tables.
    imp_customer_files = []
    for path in DQ_FILES["imperative"]:

        text = read_file(path)
        if not text:
            continue

        # Customer tables across Bronze, Silver, and Gold.
        if re.search(r'"bronze_customer"', text) or re.search(r'"silver_customer"', text) or re.search(r'"gold_dim_customer"', text):
            imp_customer_files.append(path)

    imp_customer_files = sorted(set(imp_customer_files))
    imp_files_to_modify = len(imp_customer_files)

    avg_loc_check = avg_loc_per_check_imperative()

    # Per file: one new issue_frame or bronze_metrics block plus possible union updates, approximated as two touch points.
    imp_touch_points = imp_files_to_modify * 2
    imp_loc_change = imp_touch_points * avg_loc_check

    return [
        {
            "paradigm": "declarative",
            "change_scenario": "add_email_format_valid_to_all_customer_tables",
            "files_to_modify": decl_files_to_modify,
            "estimated_loc_change": decl_loc_change,
            "touch_points": decl_touch_points,
            "complexity_score": decl_files_to_modify * decl_touch_points,
        },
        {
            "paradigm": "imperative",
            "change_scenario": "add_email_format_valid_to_all_customer_tables",
            "files_to_modify": imp_files_to_modify,
            "estimated_loc_change": imp_loc_change,
            "touch_points": imp_touch_points,
            "complexity_score": imp_files_to_modify * imp_touch_points,
        },
    ]


def change_impact_key_completeness():
    """
    Scenario: modify the existing KEY_COMPLETENESS rule group via Bronze metrics.

    Declarative approach:
      - Bronze observability joins dq_bronze_dp_metrics_overview with dq_rule_catalog for layer='bronze'.
      - The impact affects catalog RULES entries and potentially dq_bronze_dp_metrics_overview logic.

    Imperative approach:
      - Bronze metrics are implemented in 01_ip_bronze_quality.py via bronze_metrics(...).
    """

    avg_loc_check = avg_loc_per_check_imperative()

    # --- Declarative approach: KEY_COMPLETENESS entries in the rule catalog and Bronze metrics view ---

    decl_touch_points = 0
    decl_files_to_modify = set()

    # 1) Rule Catalog: RULES mit rule_group == "KEY_COMPLETENESS"
    for path in DQ_FILES["declarative"]:
        text = read_file(path)
        if not text:
            continue

        # RULES-Array in 00_dq_rule_catalog-1.py enthält z.B. {"rule_group": "KEY_COMPLETENESS", ...}
        matches_catalog = re.findall(r'"rule_group"\s*:\s*"KEY_COMPLETENESS"', text)
        if matches_catalog:
            decl_files_to_modify.add(path)
            decl_touch_points += len(matches_catalog)

        # 2) dq_bronze_dp_metrics_overview / dq_bronze_dp_observability:
        # Completeness definition changes in aggregate_metrics would affect this view.
        if "dq_bronze_dp_metrics_overview" in text or "dq_bronze_dp_observability" in text:
            # Count this as one additional touch point for the function or view definition.
            decl_files_to_modify.add(path)
            decl_touch_points += 1

    decl_result = {
        "paradigm": "declarative",
        "change_scenario": "modify_existing_rule_group_KEY_COMPLETENESS",
        "files_to_modify": len(decl_files_to_modify),
        # Approximation: each KEY_COMPLETENESS occurrence plus one touch point for the Bronze metrics view.
        "estimated_loc_change": decl_touch_points * avg_loc_check,
        "touch_points": decl_touch_points,
        "complexity_score": len(decl_files_to_modify) * max(decl_touch_points, 1) if decl_files_to_modify else 0,
    }

    # --- Imperative approach: KEY_COMPLETENESS via bronze_metrics calls ---

    bronze_files = [p for p in DQ_FILES["imperative"] if "bronze_quality" in p]
    imp_touch_points = 0
    imp_files_to_modify = set()

    for path in bronze_files:
        text = read_file(path)
        if not text:
            continue
        # Each bronze_metrics call represents a KEY_COMPLETENESS or DUPLICATE_MONITORING bundle.
        matches = re.findall(r"bronze_metrics\s*\(", text)
        if matches:
            imp_files_to_modify.add(path)
            imp_touch_points += len(matches)

    imp_result = {
        "paradigm": "imperative",
        "change_scenario": "modify_existing_rule_group_KEY_COMPLETENESS",
        "files_to_modify": len(imp_files_to_modify),
        "estimated_loc_change": imp_touch_points * avg_loc_check,
        "touch_points": imp_touch_points,
        "complexity_score": len(imp_files_to_modify) * max(imp_touch_points, 1) if imp_files_to_modify else 0,
    }

    return [decl_result, imp_result]


def change_impact_silver_001():
    """
    Scenario: modify the existing SILVER_001 rule.

    Declarative approach:
      - Locate the corresponding DP_SILVER_00x rule in the rule catalog, for example DP_SILVER_001.

    Imperative approach:
      - Locate the corresponding IP_SILVER_001 implementation in 02_ip_silver_quality.py.
    """
    # Declarative approach: search for DP_SILVER_001 in the declarative code.
    decl_touch_points = 0
    decl_files = set()
    for path in DQ_FILES["declarative"]:
        text = read_file(path)
        if not text:
            continue
        matches = re.findall(r'"DP_SILVER_001"', text)
        if matches:
            decl_files.add(path)
            decl_touch_points += len(matches)

    # Imperative approach: search for IP_SILVER_001 in Silver quality files.
    imp_touch_points = 0
    imp_files = set()
    for path in DQ_FILES["imperative"]:
        if "silver_quality" not in path:
            continue
        text = read_file(path)
        if not text:
            continue
        matches = re.findall(r'"IP_SILVER_001"', text)
        if matches:
            imp_files.add(path)
            imp_touch_points += len(matches)

    avg_loc_check = avg_loc_per_check_imperative()

    decl_result = {
        "pipeline": "declarative",
        "change_scenario": "modify_existing_rule_SILVER_001",
        "files_to_modify": len(decl_files),
        "estimated_sloc_change": decl_touch_points * avg_loc_check,
        "touch_points": decl_touch_points,
        "complexity_score": len(decl_files) * max(decl_touch_points, 1) if decl_files else 0,
    }

    imp_result = {
        "pipeline": "imperative",
        "change_scenario": "modify_existing_rule_SILVER_001",
        "files_to_modify": len(imp_files),
        "estimated_sloc_change": imp_touch_points * avg_loc_check,
        "touch_points": imp_touch_points,
        "complexity_score": len(imp_files) * max(imp_touch_points, 1) if imp_files else 0,
    }

    return [decl_result, imp_result]


# Build the final KPI 4 result table.
kpi4_rows = []

# 1) Add email_format_valid to all customers
kpi4_rows.extend(change_impact_add_rule_all_customers())

# 2) KEY_COMPLETENESS scenario for Bronze metrics in the imperative implementation.
kpi4_rows.extend(change_impact_key_completeness())

# 3) IP_SILVER_001 scenario for the Silver layer.
kpi4_rows.extend(change_impact_silver_001())

kpi4_df = pd.DataFrame(kpi4_rows)
display(kpi4_df)

## Combined KPI summary

Combines KPI 1, KPI 2, and KPI 4 into one summary table for thesis-level comparison.


In [0]:
# This cell combines the calculated KPIs into one comparison-ready summary table.

# Build a summary DataFrame that combines all KPIs in Pandas.

summary_rows = []

# Convert KPI 4 results to Pandas if necessary.
try:
    if "kpi4_df" in globals() and hasattr(kpi4_df, "toPandas"):
        kpi4_pdf = kpi4_df.toPandas()
    else:
        kpi4_pdf = kpi4_df if "kpi4_df" in globals() else pd.DataFrame()
except Exception:
    kpi4_pdf = kpi4_df if "kpi4_df" in globals() else pd.DataFrame()

k4_pdf = kpi4_pdf if "kpi4_pdf" in globals() else (kpi4_df if "kpi4_df" in globals() else pd.DataFrame())

for paradigm in PIPELINES.keys():
    # KPI 1: pipeline-level aggregation across all layers.
    k1 = kpi1_pipeline_df[kpi1_pipeline_df["pipeline"] == paradigm].iloc[0]

    # KPI 2: rule reuse and duplication metrics.
    k2 = kpi2_df[kpi2_df["paradigm"] == paradigm].iloc[0]

    # KPI 4: average change complexity across scenarios.
    if not k4_pdf.empty and "paradigm" in k4_pdf.columns:
        k4_sub = k4_pdf[k4_pdf["paradigm"] == paradigm]
        avg_complexity = k4_sub["complexity_score"].mean() if not k4_sub.empty else 0.0
    else:
        avg_complexity = 0.0

    summary_rows.append({
        "paradigm": paradigm,
        "total_loc": k1["total_sloc"],
        "dq_specific_loc": k1["dq_sloc"],
        "dq_loc_pct": k1["dq_sloc_percentage"],
        "num_files": k1["num_files"],
        "num_dq_files": k1["num_dq_files"],
        # Add pipeline-level reusable-function counts here if needed later.
        "unique_rules": k2["unique_rules"],
        "total_rule_applications": k2["total_rule_applications"],
        "reuse_ratio": k2["reuse_ratio"],
        "duplicate_blocks": k2["duplicate_blocks"],
        "duplicate_instance_ratio": k2["duplicate_instance_ratio"],
        "avg_change_complexity_score": avg_complexity,
    })

summary_df = pd.DataFrame(summary_rows)
summary_sdf = spark.createDataFrame(summary_df)
display(summary_sdf)

## KPI 1 layer-level helper table

Reformats KPI 1 output by paradigm and layer so it can be used directly for plotting.


In [0]:
# This helper table reshapes KPI 1 results into a layer-by-paradigm format for visualization.

# Add helper column that derives the paradigm from the pipeline name.
df_metrics_file["paradigm"] = df_metrics_file["pipeline"].apply(
    lambda p: "declarative" if str(p).lower().startswith("dec") else "imperative"
)

kpi1_paradigm_layer_rows = []

grouped_paradigm_layer = df_metrics_file.groupby(["paradigm", "layer"], dropna=False)

for (paradigm, layer), group in grouped_paradigm_layer:
    total_sloc = group["sloc"].sum()
    dq_sloc = group.loc[group["is_data_quality"], "sloc"].sum()
    dq_percentage = (dq_sloc / total_sloc * 100.0) if total_sloc > 0 else 0.0

    num_files = group["file_path"].nunique()
    num_dq_files = group.loc[group["is_data_quality"], "file_path"].nunique()

    kpi1_paradigm_layer_rows.append(
        {
            "paradigm": paradigm,
            "layer": layer,
            "total_sloc": total_sloc,
            "dq_sloc": dq_sloc,
            "dq_sloc_percentage": dq_percentage,
            "num_files": num_files,
            "num_dq_files": num_dq_files,
        }
    )

kpi1_paradigm_layer_df = (
    pd.DataFrame(kpi1_paradigm_layer_rows)
    .sort_values(["layer", "paradigm"])
    .reset_index(drop=True)
)

print("KPI 1 – DQ-specific SLOC percentage by paradigm and layer:")
display(kpi1_paradigm_layer_df)


## Visualize data quality KPIs

Creates comparison charts for DQ code share, duplicate-block ratio, and change complexity.


In [0]:
# This plotting cell creates figures that summarize the main DQ KPI findings.

# Assumption: the notebook defines consistent colors for both paradigms beforehand.
# declarative_color = "#1f77b4"
# imperative_color = "#ff7f0e"
# Any consistent color definition from the first analysis dimension can be reused here.

# --------------------------------------------------------------------
# Figure 1 – Data Quality Code Share (%) per Paradigm (KPI 1)
# --------------------------------------------------------------------
layer_order = ["catalog", "bronze", "silver", "gold"]

# Keep only existing layers while preserving the intended medallion order.
layers = [l for l in layer_order if l in kpi1_paradigm_layer_df["layer"].unique()]
plot_df = (
    kpi1_paradigm_layer_df
    .pivot(index="layer", columns="paradigm", values="dq_sloc_percentage")
    .reindex(layers)
)

x = np.arange(len(layers))
width = 0.35

fig, ax = plt.subplots(figsize=(6, 4))

dec_vals = plot_df.get("declarative", pd.Series(0, index=layers)).values
imp_vals = plot_df.get("imperative", pd.Series(0, index=layers)).values

ax.bar(x - width/2, dec_vals, width, label="Declarative", color=declarative_color)
ax.bar(x + width/2, imp_vals, width, label="Imperative", color=imperative_color)

ax.set_xticks(x)
ax.set_xticklabels(layers)
ax.set_ylabel("DQ-specific SLOC (%)")

ax.set_ylim(0, max(dec_vals.max(), imp_vals.max()) * 1.1 if len(layers) > 0 else 1)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.legend()
plt.tight_layout()
plt.show()



# --------------------------------------------------------------------
# Figure 2 – Duplicate Code Blocks per Paradigm (KPI 2)
# --------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 4))

x = np.arange(len(kpi2_df["paradigm"]))
width = 0.6
dup_vals = kpi2_df["duplicate_block_ratio"].values

colors = []
for p in kpi2_df["paradigm"]:
    if p.lower().startswith("dec"):
        colors.append(declarative_color)
    else:
        colors.append(imperative_color)

ax.bar(x, dup_vals, width, color=colors)

ax.set_xticks(x)
ax.set_xticklabels(kpi2_df["paradigm"])
ax.set_ylabel("Duplicate block ratio (%)")

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()


# --------------------------------------------------------------------
# Figure 5 – Change Complexity Across Scenarios (KPI 4)
# --------------------------------------------------------------------
# Expected input: kpi4_df with change_scenario, paradigm, and complexity_score columns.

scenario_label_map = {
    "add_email_format_valid_to_all_customer_tables": "add format validation",
    "modify_existing_rule_group_KEY_COMPLETENESS": "modify rule group",
    "modify_existing_rule_SILVER_001": "modify rule",
}

scenarios_sorted = list(kpi4_df["change_scenario"].unique())

x = np.arange(len(scenarios_sorted))
width = 0.35

dec_vals = []
imp_vals = []
xtick_labels = []

for scen in scenarios_sorted:
    sub = kpi4_df[kpi4_df["change_scenario"] == scen]

    dec_row = sub[sub["paradigm"].str.lower().str.startswith("dec")]
    imp_row = sub[sub["paradigm"].str.lower().str.startswith("imp")]

    dec_vals.append(dec_row["complexity_score"].iloc[0] if not dec_row.empty else 0.0)
    imp_vals.append(imp_row["complexity_score"].iloc[0] if not imp_row.empty else 0.0)

    xtick_labels.append(scenario_label_map.get(scen, scen))

fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(x - width / 2, dec_vals, width, label="Declarative", color=declarative_color)
ax.bar(x + width / 2, imp_vals, width, label="Imperative", color=imperative_color)

ax.set_xticks(x)
ax.set_xticklabels(xtick_labels, rotation=20, ha="right")
ax.set_ylabel("Complexity score")
ax.legend()

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()